# 📈 Stock News Sentiment + Price Movement Predictor

> **⚠️ Research & Analytics Disclaimer**: This notebook is for **educational and research purposes only**. It does **not** constitute financial advice, investment recommendations, or trading signals. Past sentiment-price correlations do not guarantee future results. Always consult a licensed financial advisor before making investment decisions.

---

## 🎯 Project Overview

This end-to-end machine learning project explores the relationship between **financial news sentiment** and **next-day stock price movements**. We'll build a complete NLP + classification pipeline covering:

| Stage | What We Do |
|-------|------------|
| 1. Data Collection | Fetch historical stock prices via `yfinance` + news headlines |
| 2. Sentiment Analysis | Use FinBERT (finance-tuned BERT) to score news sentiment |
| 3. Feature Engineering | Combine sentiment scores with technical price features |
| 4. ML Classification | Predict next-day UP/DOWN/NEUTRAL movement |
| 5. Evaluation | Confusion matrix, classification report, feature importance |
| 6. Visualization | Rich interactive charts and dashboards |

### ML Concepts Demonstrated
- **NLP**: Tokenization, transformer embeddings, sentiment classification
- **Time-Series Features**: Rolling windows, lag features, momentum indicators
- **Binary/Multi-class Classification**: Random Forest, Gradient Boosting
- **Data Visualization**: Matplotlib, Seaborn, Plotly charts
- **Model Evaluation**: Cross-validation, precision/recall, ROC curves

## 📦 Section 1: Installation & Setup

We install all required libraries. Key dependencies:
- **`yfinance`**: Yahoo Finance API wrapper for historical OHLCV data
- **`transformers`**: Hugging Face library for FinBERT model
- **`torch`**: PyTorch backend for transformer inference
- **`scikit-learn`**: ML algorithms and evaluation metrics
- **`plotly`**: Interactive visualizations
- **`newsapi-python`**: Optional — fetch real headlines (requires free API key)

In [ ]:
# Install all required packages
# Run this cell once; restart kernel after if needed
!pip install yfinance transformers torch scikit-learn pandas numpy matplotlib seaborn plotly newsapi-python requests tqdm --quiet

## 🔧 Section 2: Imports & Configuration

We organize imports by category and define global configuration constants.

In [ ]:
# ── Standard Library ──────────────────────────────────────────────────────────
import warnings
import random
import json
from datetime import datetime, timedelta
from pathlib import Path

warnings.filterwarnings('ignore')

# ── Data & Math ───────────────────────────────────────────────────────────────
import numpy as np
import pandas as pd

# ── Finance Data ──────────────────────────────────────────────────────────────
import yfinance as yf

# ── NLP / Hugging Face ────────────────────────────────────────────────────────
from transformers import AutoTokenizer, AutoModelForSequenceClassification
import torch
import torch.nn.functional as F

# ── Machine Learning ──────────────────────────────────────────────────────────
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split, cross_val_score, StratifiedKFold
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.metrics import (
    classification_report, confusion_matrix, roc_auc_score,
    accuracy_score, ConfusionMatrixDisplay
)
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer

# ── Visualization ─────────────────────────────────────────────────────────────
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
import plotly.graph_objects as go
import plotly.express as px
from plotly.subplots import make_subplots

# ── Utilities ─────────────────────────────────────────────────────────────────
from tqdm import tqdm

# ── Configuration ─────────────────────────────────────────────────────────────
CONFIG = {
    # Stock universe to analyze
    'tickers': ['AAPL', 'MSFT', 'GOOGL', 'TSLA', 'AMZN'],
    
    # Historical lookback period (yfinance)
    'start_date': '2023-01-01',
    'end_date': datetime.today().strftime('%Y-%m-%d'),
    
    # Price movement thresholds for labeling
    'up_threshold': 0.005,    # >+0.5% → UP
    'down_threshold': -0.005, # <-0.5% → DOWN
    # Between -0.5% and +0.5% → NEUTRAL
    
    # FinBERT model (finance-domain BERT)
    'model_name': 'ProsusAI/finbert',
    
    # ML hyperparameters
    'test_size': 0.2,
    'random_state': 42,
    'cv_folds': 5,
    
    # Rolling window sizes for technical features
    'windows': [3, 7, 14, 30],
}

# Plotting style
plt.rcParams.update({
    'figure.facecolor': '#0f1117',
    'axes.facecolor': '#1a1d2e',
    'axes.edgecolor': '#2d3057',
    'axes.labelcolor': '#e0e0e0',
    'xtick.color': '#a0a0b0',
    'ytick.color': '#a0a0b0',
    'text.color': '#e0e0e0',
    'grid.color': '#2d3057',
    'grid.alpha': 0.5,
    'figure.dpi': 120,
})

COLORS = {
    'up': '#00d4aa',
    'down': '#ff4d6d',
    'neutral': '#f5a623',
    'positive': '#00d4aa',
    'negative': '#ff4d6d',
    'accent': '#7b68ee',
    'bg': '#0f1117',
}

print('✅ All imports successful!')
print(f'📅 Analysis period: {CONFIG["start_date"]} → {CONFIG["end_date"]}')
print(f'📊 Tickers: {CONFIG["tickers"]}')
print(f'🤖 NLP Model: {CONFIG["model_name"]}')

## 📰 Section 3: News Data

### 3.1 Understanding News Sources

In production, you'd connect to:
- **NewsAPI** (newsapi.org) — free tier: 100 req/day
- **Alpha Vantage News API** — financial news with tickers
- **Finnhub** — real-time news with sentiment
- **RSS Feeds** — Bloomberg, Reuters, CNBC

For this notebook, we generate **realistic synthetic headlines** that mirror actual financial news patterns. Each headline follows templates used by real financial journalists.

### 3.2 Why Synthetic Data?
- No API key required
- Reproducible experiments
- Controlled sentiment distribution
- Immediate demonstration of the full pipeline

The Streamlit app includes a live NewsAPI integration toggle.

In [ ]:
def generate_realistic_headlines(ticker: str, n_days: int = 365) -> pd.DataFrame:
    """
    Generate realistic financial news headlines for a given ticker.
    
    Templates are drawn from actual headline patterns observed in
    Bloomberg, Reuters, WSJ, and CNBC financial reporting.
    
    Returns a DataFrame with columns: date, headline, ticker
    """
    
    # Company name mapping
    names = {
        'AAPL': 'Apple', 'MSFT': 'Microsoft', 'GOOGL': 'Alphabet',
        'TSLA': 'Tesla', 'AMZN': 'Amazon', 'META': 'Meta',
        'NVDA': 'Nvidia', 'JPM': 'JPMorgan'
    }
    name = names.get(ticker, ticker)
    
    # ── POSITIVE headline templates ────────────────────────────────────────────
    positive_templates = [
        f"{name} beats Q{{q}} earnings estimates by {{pct}}%, shares surge",
        f"{name} reports record revenue of ${{rev}}B, analysts raise price targets",
        f"{name} announces ${{bb}}B share buyback program, boosting investor confidence",
        f"{name} expands into {{market}} market, analysts see significant upside",
        f"Strong demand for {name} products drives quarterly profit above forecasts",
        f"{name} CEO unveils AI strategy, Wall Street responds positively",
        f"{name} gross margins improve to {{margin}}%, signaling operational efficiency",
        f"Analysts upgrade {name} to 'Buy' citing strong growth trajectory",
        f"{name} secures major government contract worth ${{val}}B",
        f"{name} partnership with {{partner}} seen as game-changer for revenue growth",
        f"{name} raises full-year guidance, stock climbs on optimism",
        f"Institutional investors increase {name} holdings by {{pct}}% this quarter",
    ]
    
    # ── NEGATIVE headline templates ────────────────────────────────────────────
    negative_templates = [
        f"{name} misses Q{{q}} revenue expectations amid weakening demand",
        f"{name} faces regulatory scrutiny in {{region}}, shares fall sharply",
        f"{name} lowers full-year guidance, citing macroeconomic headwinds",
        f"Supply chain disruptions weigh heavily on {name} profit margins",
        f"{name} announces {{n}}K layoffs as part of cost-cutting restructuring",
        f"Class action lawsuit filed against {name} over data privacy violations",
        f"Analysts downgrade {name} to 'Sell' on slowing growth concerns",
        f"{name} product recall hits revenue, stock drops {{pct}}%",
        f"Competition intensifies as rivals undercut {name}'s pricing strategy",
        f"{name} CFO resigns unexpectedly, raising governance concerns",
        f"Rising interest rates pressure {name}'s debt-heavy balance sheet",
        f"{name} loses major client, analysts slash price targets significantly",
    ]
    
    # ── NEUTRAL headline templates ─────────────────────────────────────────────
    neutral_templates = [
        f"{name} to release Q{{q}} earnings on {{month}} {{day}}",
        f"{name} holds annual shareholder meeting, discusses strategic priorities",
        f"{name} files 10-Q with SEC, details operational highlights",
        f"{name} appoints new board member with extensive industry experience",
        f"{name} stock trades near 52-week average as markets consolidate",
        f"Analysts maintain 'Hold' rating on {name} ahead of earnings",
        f"{name} participates in industry conference, reiterates guidance",
        f"{name} confirms dividend payment schedule for upcoming quarter",
        f"Trading volume in {name} remains steady ahead of Fed decision",
        f"{name} CEO speaks at tech summit, outlines 5-year vision",
    ]
    
    records = []
    base_date = datetime.today() - timedelta(days=n_days)
    
    for i in range(n_days):
        date = base_date + timedelta(days=i)
        
        # Skip weekends (markets closed)
        if date.weekday() >= 5:
            continue
        
        # Generate 1-4 headlines per trading day
        n_headlines = random.choices([1, 2, 3, 4], weights=[0.3, 0.4, 0.2, 0.1])[0]
        
        for _ in range(n_headlines):
            # Weighted sentiment distribution (realistic: ~50% neutral, 30% pos, 20% neg)
            sentiment_type = random.choices(
                ['positive', 'negative', 'neutral'],
                weights=[0.30, 0.20, 0.50]
            )[0]
            
            if sentiment_type == 'positive':
                template = random.choice(positive_templates)
            elif sentiment_type == 'negative':
                template = random.choice(negative_templates)
            else:
                template = random.choice(neutral_templates)
            
            # Fill template placeholders
            headline = template.format(
                q=random.randint(1, 4),
                pct=round(random.uniform(2, 15), 1),
                rev=round(random.uniform(10, 150), 1),
                bb=round(random.uniform(5, 50), 0),
                val=round(random.uniform(1, 20), 1),
                market=random.choice(['cloud', 'AI', 'healthcare', 'automotive', 'fintech']),
                margin=round(random.uniform(35, 65), 1),
                region=random.choice(['EU', 'US', 'China', 'India', 'Brazil']),
                n=random.choice([1, 2, 5, 10, 15]),
                partner=random.choice(['OpenAI', 'Samsung', 'AWS', 'Oracle', 'Salesforce']),
                month=random.choice(['January', 'April', 'July', 'October']),
                day=random.randint(15, 28),
            )
            
            records.append({
                'date': date.strftime('%Y-%m-%d'),
                'headline': headline,
                'ticker': ticker,
                '_true_sentiment': sentiment_type  # Ground truth for validation
            })
    
    df = pd.DataFrame(records)
    df['date'] = pd.to_datetime(df['date'])
    return df


# Generate headlines for all tickers
print('📰 Generating realistic financial news headlines...')
news_dfs = []
for ticker in CONFIG['tickers']:
    df = generate_realistic_headlines(ticker, n_days=400)
    news_dfs.append(df)
    print(f'  {ticker}: {len(df)} headlines across {df["date"].nunique()} trading days')

news_df = pd.concat(news_dfs, ignore_index=True)
print(f'\n✅ Total headlines generated: {len(news_df):,}')
print(f'📊 Date range: {news_df["date"].min().date()} → {news_df["date"].max().date()}')
news_df.head(10)

## 📈 Section 4: Stock Price Data via yfinance

### Understanding OHLCV Data

Each row from yfinance represents one trading day:
- **Open**: Price at market open (9:30 AM ET)
- **High**: Intraday highest price
- **Low**: Intraday lowest price  
- **Close**: Adjusted closing price (accounts for splits/dividends)
- **Volume**: Total shares traded that day

We compute the **next-day return** as our prediction target:
```
return_t+1 = (Close_{t+1} - Close_t) / Close_t
```

Then we label it:
- return > +0.5% → **UP** ↑
- return < -0.5% → **DOWN** ↓
- otherwise → **NEUTRAL** →

In [ ]:
def fetch_and_engineer_prices(ticker: str) -> pd.DataFrame:
    """
    Fetch historical OHLCV data and engineer technical features.
    
    Technical features are common signals used by quant analysts:
    - Returns (daily, multi-day)
    - Moving averages (SMA, EMA)
    - Volatility (rolling std of returns)
    - Volume momentum
    - RSI (Relative Strength Index)
    - Bollinger Band position
    """
    print(f'  Fetching {ticker}...')
    
    # Download adjusted close prices
    df = yf.download(
        ticker,
        start=CONFIG['start_date'],
        end=CONFIG['end_date'],
        progress=False
    )
    
    if df.empty:
        print(f'    ⚠️ No data for {ticker}')
        return pd.DataFrame()
    
    # Flatten MultiIndex columns if present
    if isinstance(df.columns, pd.MultiIndex):
        df.columns = df.columns.get_level_values(0)
    
    df = df.reset_index()
    df['ticker'] = ticker
    df.columns = [c.lower().replace(' ', '_') for c in df.columns]
    
    # ── Target Variable ────────────────────────────────────────────────────────
    df['return_1d'] = df['close'].pct_change()          # Today's return
    df['next_return'] = df['return_1d'].shift(-1)       # Next day's return (TARGET)
    
    # Label: UP / DOWN / NEUTRAL
    def label_movement(ret):
        if pd.isna(ret):
            return np.nan
        elif ret > CONFIG['up_threshold']:
            return 'UP'
        elif ret < CONFIG['down_threshold']:
            return 'DOWN'
        else:
            return 'NEUTRAL'
    
    df['label'] = df['next_return'].apply(label_movement)
    
    # ── Technical Features ─────────────────────────────────────────────────────
    
    # 1. Multi-day returns (momentum)
    for w in CONFIG['windows']:
        df[f'return_{w}d'] = df['close'].pct_change(w)
    
    # 2. Simple Moving Averages
    for w in CONFIG['windows']:
        df[f'sma_{w}'] = df['close'].rolling(w).mean()
        df[f'price_vs_sma_{w}'] = df['close'] / df[f'sma_{w}'] - 1  # % above/below SMA
    
    # 3. Exponential Moving Average
    df['ema_12'] = df['close'].ewm(span=12).mean()
    df['ema_26'] = df['close'].ewm(span=26).mean()
    df['macd'] = df['ema_12'] - df['ema_26']            # MACD signal
    df['macd_signal'] = df['macd'].ewm(span=9).mean()
    df['macd_hist'] = df['macd'] - df['macd_signal']
    
    # 4. Volatility (rolling std of daily returns)
    for w in CONFIG['windows']:
        df[f'volatility_{w}d'] = df['return_1d'].rolling(w).std()
    
    # 5. Volume features
    df['volume_ma_10'] = df['volume'].rolling(10).mean()
    df['volume_ratio'] = df['volume'] / df['volume_ma_10']   # Relative volume
    df['volume_return'] = df['volume'].pct_change()
    
    # 6. RSI (Relative Strength Index) — overbought/oversold indicator
    delta = df['close'].diff()
    gain = delta.clip(lower=0).rolling(14).mean()
    loss = (-delta.clip(upper=0)).rolling(14).mean()
    rs = gain / loss.replace(0, np.nan)
    df['rsi_14'] = 100 - (100 / (1 + rs))
    
    # 7. Bollinger Bands (price position within volatility envelope)
    bb_ma = df['close'].rolling(20).mean()
    bb_std = df['close'].rolling(20).std()
    df['bb_upper'] = bb_ma + 2 * bb_std
    df['bb_lower'] = bb_ma - 2 * bb_std
    df['bb_position'] = (df['close'] - df['bb_lower']) / (df['bb_upper'] - df['bb_lower'])
    
    # 8. High-Low range (intraday volatility)
    df['hl_range'] = (df['high'] - df['low']) / df['close']
    
    # 9. Gap feature (overnight price change)
    df['overnight_gap'] = (df['open'] - df['close'].shift(1)) / df['close'].shift(1)
    
    # 10. Lag features (yesterday's return as a feature)
    for lag in [1, 2, 3, 5]:
        df[f'return_lag_{lag}'] = df['return_1d'].shift(lag)
    
    print(f'    ✅ {len(df)} rows, {len(df.columns)} features')
    return df


# Fetch all tickers
print('📈 Fetching stock price data from Yahoo Finance...')
price_dfs = []
for ticker in CONFIG['tickers']:
    df = fetch_and_engineer_prices(ticker)
    if not df.empty:
        price_dfs.append(df)

prices_df = pd.concat(price_dfs, ignore_index=True)
print(f'\n✅ Total price records: {len(prices_df):,}')
print(f'📊 Features engineered: {len(prices_df.columns)} columns')
prices_df[['date', 'ticker', 'close', 'return_1d', 'next_return', 'label', 'rsi_14', 'macd']].head()

## 🤖 Section 5: FinBERT Sentiment Analysis

### What is FinBERT?

**FinBERT** is a BERT-based model fine-tuned on financial text by researchers at Prosus AI. It outperforms generic sentiment models on financial news because:

1. **Domain vocabulary**: Financial terms like "headwinds", "beat estimates", "guidance" have specific financial meanings that general NLP models misinterpret
2. **Context understanding**: "Volatility" in finance is neutral (not negative); FinBERT knows this
3. **Training data**: Pre-trained on financial news, analyst reports, earnings calls

### Output Format
FinBERT returns 3 probability scores:
```
{'positive': 0.87, 'negative': 0.08, 'neutral': 0.05}
```

We use these as continuous features AND derive a composite sentiment score:
```
sentiment_score = positive_prob - negative_prob ∈ [-1, +1]
```

In [ ]:
class FinBERTAnalyzer:
    """
    Wrapper around the FinBERT model for financial news sentiment analysis.
    
    Uses ProsusAI/finbert which is fine-tuned specifically on financial text.
    Handles batching and GPU/CPU device selection automatically.
    """
    
    def __init__(self, model_name: str = 'ProsusAI/finbert'):
        print(f'🤖 Loading {model_name}...')
        self.tokenizer = AutoTokenizer.from_pretrained(model_name)
        self.model = AutoModelForSequenceClassification.from_pretrained(model_name)
        
        # Use GPU if available, else CPU
        self.device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
        self.model = self.model.to(self.device)
        self.model.eval()
        
        # Label order from FinBERT config
        self.labels = ['positive', 'negative', 'neutral']
        print(f'✅ FinBERT loaded on {self.device}')
    
    def analyze_batch(self, texts: list, batch_size: int = 16) -> pd.DataFrame:
        """
        Analyze a list of texts in batches for efficiency.
        
        Args:
            texts: List of headline strings
            batch_size: Number of texts per forward pass
            
        Returns:
            DataFrame with columns: positive, negative, neutral, 
                                    sentiment_label, sentiment_score
        """
        all_results = []
        
        for i in tqdm(range(0, len(texts), batch_size), desc='🔍 FinBERT inference'):
            batch = texts[i : i + batch_size]
            
            # Tokenize: truncate to 512 tokens (BERT limit), pad to batch max length
            inputs = self.tokenizer(
                batch,
                padding=True,
                truncation=True,
                max_length=512,
                return_tensors='pt'
            ).to(self.device)
            
            # Forward pass (no gradient computation needed for inference)
            with torch.no_grad():
                outputs = self.model(**inputs)
            
            # Convert logits → probabilities via softmax
            probs = F.softmax(outputs.logits, dim=-1).cpu().numpy()
            
            for prob_row in probs:
                result = dict(zip(self.labels, prob_row))
                
                # Composite score: positive - negative ∈ [-1, +1]
                result['sentiment_score'] = float(result['positive'] - result['negative'])
                
                # Dominant label
                result['sentiment_label'] = max(
                    ['positive', 'negative', 'neutral'],
                    key=lambda k: result[k]
                )
                all_results.append(result)
        
        return pd.DataFrame(all_results)
    
    def analyze_single(self, text: str) -> dict:
        """Analyze a single headline (convenience wrapper)."""
        result = self.analyze_batch([text])
        return result.iloc[0].to_dict()


# Initialize FinBERT
analyzer = FinBERTAnalyzer(CONFIG['model_name'])

# Quick sanity check with known examples
test_headlines = [
    "Apple beats earnings estimates by 12%, shares surge to all-time high",
    "Microsoft faces antitrust probe in Europe, shares fall 4%",
    "Tesla announces annual shareholder meeting date for next month",
    "Amazon revenue misses Wall Street expectations amid slowing consumer spending",
    "Nvidia raises guidance citing unprecedented AI chip demand",
]

print('\n🧪 Sentiment Sanity Check:')
print('=' * 80)
test_results = analyzer.analyze_batch(test_headlines, batch_size=5)
for headline, row in zip(test_headlines, test_results.itertuples()):
    emoji = {'positive': '🟢', 'negative': '🔴', 'neutral': '🟡'}[row.sentiment_label]
    print(f'{emoji} [{row.sentiment_label.upper():8s}] score={row.sentiment_score:+.3f} | {headline[:60]}')

In [ ]:
# Run FinBERT on all generated headlines
print(f'🔍 Running FinBERT on {len(news_df):,} headlines...')
print('   (This takes 2-5 minutes on CPU, ~30s on GPU)')

sentiment_results = analyzer.analyze_batch(
    news_df['headline'].tolist(),
    batch_size=32
)

# Merge sentiment results with news_df
news_sentiment_df = pd.concat([
    news_df.reset_index(drop=True),
    sentiment_results.reset_index(drop=True)
], axis=1)

print(f'\n✅ Sentiment analysis complete!')
print('\n📊 Sentiment Distribution:')
sent_dist = news_sentiment_df['sentiment_label'].value_counts(normalize=True) * 100
for label, pct in sent_dist.items():
    bar = '█' * int(pct / 2)
    print(f'  {label:8s}: {bar} {pct:.1f}%')

print('\n📈 Sentiment Score Statistics:')
print(news_sentiment_df['sentiment_score'].describe().round(4))

news_sentiment_df.head()

## 🔗 Section 6: Aggregating Sentiment by Date & Ticker

Since we have **multiple headlines per day per stock**, we aggregate them into **daily sentiment features**:

| Feature | Description |
|---------|-------------|
| `mean_sentiment_score` | Average composite score for the day |
| `max_sentiment_score` | Most extreme positive headline |
| `min_sentiment_score` | Most extreme negative headline |
| `n_positive` | Count of positive headlines |
| `n_negative` | Count of negative headlines |
| `positive_ratio` | Fraction of headlines that were positive |
| `sentiment_std` | Sentiment disagreement across headlines |
| `rolling_sentiment_*d` | Rolling average of past N days' sentiment |

**Why aggregate?** Individual headlines are noisy; the *pattern across headlines* on a given day carries more signal.

In [ ]:
def aggregate_daily_sentiment(df: pd.DataFrame) -> pd.DataFrame:
    """
    Aggregate per-headline sentiment into per-day, per-ticker features.
    """
    agg = df.groupby(['date', 'ticker']).agg(
        n_headlines=('headline', 'count'),
        mean_sentiment_score=('sentiment_score', 'mean'),
        max_sentiment_score=('sentiment_score', 'max'),
        min_sentiment_score=('sentiment_score', 'min'),
        std_sentiment_score=('sentiment_score', 'std'),
        mean_positive_prob=('positive', 'mean'),
        mean_negative_prob=('negative', 'mean'),
        mean_neutral_prob=('neutral', 'mean'),
        n_positive=('sentiment_label', lambda x: (x == 'positive').sum()),
        n_negative=('sentiment_label', lambda x: (x == 'negative').sum()),
        n_neutral=('sentiment_label', lambda x: (x == 'neutral').sum()),
    ).reset_index()
    
    # Derived ratios
    agg['positive_ratio'] = agg['n_positive'] / agg['n_headlines']
    agg['negative_ratio'] = agg['n_negative'] / agg['n_headlines']
    agg['sentiment_balance'] = agg['positive_ratio'] - agg['negative_ratio']
    
    # Rolling sentiment features (within each ticker)
    agg = agg.sort_values(['ticker', 'date'])
    for window in [3, 7, 14]:
        agg[f'rolling_sentiment_{window}d'] = (
            agg.groupby('ticker')['mean_sentiment_score']
               .transform(lambda x: x.rolling(window, min_periods=1).mean())
        )
        agg[f'sentiment_momentum_{window}d'] = (
            agg.groupby('ticker')['mean_sentiment_score']
               .transform(lambda x: x.rolling(window, min_periods=1).mean())
               - agg.groupby('ticker')['mean_sentiment_score']
                    .transform(lambda x: x.shift(window))
        )
    
    # Lag features (yesterday's sentiment)
    for lag in [1, 2, 3]:
        agg[f'sentiment_lag_{lag}d'] = (
            agg.groupby('ticker')['mean_sentiment_score']
               .transform(lambda x: x.shift(lag))
        )
    
    agg['std_sentiment_score'] = agg['std_sentiment_score'].fillna(0)
    
    return agg


daily_sentiment = aggregate_daily_sentiment(news_sentiment_df)
print(f'✅ Daily sentiment features: {len(daily_sentiment):,} rows × {len(daily_sentiment.columns)} features')
print(f'📅 Date range: {daily_sentiment["date"].min().date()} → {daily_sentiment["date"].max().date()}')
daily_sentiment.head()

## 🔗 Section 7: Merging Sentiment + Price Features

Now we join the sentiment features (from news) with the technical features (from prices) on **date × ticker**. This creates our final feature matrix for ML training.

In [ ]:
# Normalize date columns
prices_df['date'] = pd.to_datetime(prices_df['date'])
daily_sentiment['date'] = pd.to_datetime(daily_sentiment['date'])

# Merge on date + ticker
master_df = prices_df.merge(
    daily_sentiment,
    on=['date', 'ticker'],
    how='left'
)

print(f'✅ Master dataset: {len(master_df):,} rows × {len(master_df.columns)} columns')
print(f'📊 Label distribution (across all tickers):')
label_counts = master_df['label'].value_counts()
for label, count in label_counts.items():
    pct = count / len(master_df.dropna(subset=['label'])) * 100
    print(f'  {label:7s}: {count:4d} ({pct:.1f}%)')

# How many days have news coverage?
has_news = master_df['n_headlines'].notna().sum()
print(f'\n📰 Days with news coverage: {has_news:,} / {len(master_df):,} ({100*has_news/len(master_df):.1f}%)')

## 📊 Section 8: Exploratory Data Analysis (EDA)

Before building models, we visualize the data to:
1. Understand sentiment distributions
2. Explore sentiment-return relationships
3. Identify potential predictive signals
4. Spot data quality issues

In [ ]:
# ── Figure 1: Price Charts for All Tickers ─────────────────────────────────
fig, axes = plt.subplots(len(CONFIG['tickers']), 1, figsize=(16, 3.5 * len(CONFIG['tickers'])))
fig.patch.set_facecolor(COLORS['bg'])
fig.suptitle('📈 Stock Price History with Sentiment Overlay', 
             fontsize=18, fontweight='bold', color='white', y=1.01)

for ax, ticker in zip(axes, CONFIG['tickers']):
    ticker_data = master_df[master_df['ticker'] == ticker].dropna(subset=['close'])
    
    # Price line
    ax.plot(ticker_data['date'], ticker_data['close'], 
            color='#7b68ee', linewidth=1.5, zorder=3)
    ax.fill_between(ticker_data['date'], ticker_data['close'],
                    alpha=0.15, color='#7b68ee')
    
    # Color background by sentiment (where available)
    if 'mean_sentiment_score' in ticker_data.columns:
        sent = ticker_data['mean_sentiment_score'].fillna(0)
        for i in range(len(ticker_data) - 1):
            color = COLORS['up'] if sent.iloc[i] > 0.1 else \
                    COLORS['down'] if sent.iloc[i] < -0.1 else COLORS['neutral']
            ax.axvspan(ticker_data['date'].iloc[i], ticker_data['date'].iloc[i+1],
                       alpha=0.05, color=color)
    
    # 20-day SMA
    if 'sma_7' in ticker_data.columns:
        ax.plot(ticker_data['date'], ticker_data['sma_7'],
                color='#f5a623', linewidth=1, linestyle='--', alpha=0.7, label='SMA-7')
    
    ax.set_title(f'{ticker}', fontsize=13, fontweight='bold', color='white', pad=8)
    ax.set_ylabel('Price ($)', fontsize=10)
    ax.legend(fontsize=9)
    ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('price_history.png', dpi=120, bbox_inches='tight', facecolor=COLORS['bg'])
plt.show()
print('📁 Saved: price_history.png')

In [ ]:
# ── Figure 2: Sentiment Distribution ──────────────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(16, 5))
fig.patch.set_facecolor(COLORS['bg'])
fig.suptitle('🎭 FinBERT Sentiment Analysis', fontsize=16, fontweight='bold', color='white')

# 2a: Overall sentiment pie
ax = axes[0]
sent_counts = news_sentiment_df['sentiment_label'].value_counts()
colors_pie = [COLORS['positive'], COLORS['negative'], COLORS['neutral']]
wedges, texts, autotexts = ax.pie(
    sent_counts.values,
    labels=sent_counts.index,
    autopct='%1.1f%%',
    colors=colors_pie,
    startangle=90,
    textprops={'color': 'white', 'fontsize': 12},
    wedgeprops={'edgecolor': '#0f1117', 'linewidth': 2}
)
ax.set_title('Sentiment Label Distribution', color='white', fontweight='bold')

# 2b: Sentiment score distribution by ticker
ax = axes[1]
for i, ticker in enumerate(CONFIG['tickers']):
    ticker_sentiment = news_sentiment_df[news_sentiment_df['ticker'] == ticker]['sentiment_score']
    ax.hist(ticker_sentiment, bins=30, alpha=0.6, label=ticker, density=True)
ax.axvline(x=0, color='white', linestyle='--', alpha=0.5, linewidth=1.5)
ax.set_xlabel('Sentiment Score')
ax.set_ylabel('Density')
ax.set_title('Score Distribution by Ticker', color='white', fontweight='bold')
ax.legend(fontsize=9)
ax.grid(True, alpha=0.3)

# 2c: Sentiment vs Next-Day Return
ax = axes[2]
merged_sample = master_df.dropna(subset=['mean_sentiment_score', 'next_return', 'label'])
for label, color in [('UP', COLORS['up']), ('DOWN', COLORS['down']), ('NEUTRAL', COLORS['neutral'])]:
    subset = merged_sample[merged_sample['label'] == label]
    ax.scatter(
        subset['mean_sentiment_score'],
        subset['next_return'] * 100,
        c=color, alpha=0.3, s=15, label=label
    )
ax.axhline(y=0, color='white', linestyle='--', alpha=0.4)
ax.axvline(x=0, color='white', linestyle='--', alpha=0.4)
ax.set_xlabel('Daily Sentiment Score')
ax.set_ylabel('Next-Day Return (%)')
ax.set_title('Sentiment vs Price Movement', color='white', fontweight='bold')
ax.legend(fontsize=9)
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('sentiment_analysis.png', dpi=120, bbox_inches='tight', facecolor=COLORS['bg'])
plt.show()

In [ ]:
# ── Figure 3: Feature Correlation Heatmap ─────────────────────────────────────
feature_cols_for_corr = [
    'mean_sentiment_score', 'positive_ratio', 'negative_ratio',
    'return_1d', 'return_3d', 'return_7d', 'rsi_14', 'macd',
    'volatility_7d', 'volume_ratio', 'bb_position', 'next_return'
]

corr_df = master_df[feature_cols_for_corr].dropna()
corr_matrix = corr_df.corr()

fig, ax = plt.subplots(figsize=(14, 10))
fig.patch.set_facecolor(COLORS['bg'])
ax.set_facecolor(COLORS['bg'])

mask = np.zeros_like(corr_matrix, dtype=bool)
mask[np.triu_indices_from(mask)] = True

sns.heatmap(
    corr_matrix,
    mask=mask,
    annot=True, fmt='.2f', fontsize=9,
    cmap='RdYlGn',
    center=0, vmin=-1, vmax=1,
    ax=ax,
    linewidths=0.5,
    cbar_kws={'label': 'Pearson Correlation'}
)
ax.set_title('Feature Correlation Heatmap\n(Focus: correlation with next_return)', 
             fontsize=14, fontweight='bold', color='white', pad=15)
plt.xticks(rotation=45, ha='right', fontsize=10)
plt.yticks(fontsize=10)
plt.tight_layout()
plt.savefig('correlation_heatmap.png', dpi=120, bbox_inches='tight', facecolor=COLORS['bg'])
plt.show()

# Print top correlations with next_return
next_corr = corr_matrix['next_return'].drop('next_return').sort_values(key=abs, ascending=False)
print('\n🔍 Top Feature Correlations with Next-Day Return:')
for feat, corr in next_corr.head(10).items():
    direction = '↑' if corr > 0 else '↓'
    print(f'  {direction} {feat:35s}: {corr:+.4f}')

## 🧠 Section 9: Machine Learning Pipeline

### Model Architecture

We frame this as a **3-class classification** problem:
- **Input X**: Sentiment features + Technical indicators (as of day t)
- **Output y**: Price movement label on day t+1 (UP / DOWN / NEUTRAL)

### Why NOT regression?
Predicting the *direction* (classification) is more practically useful and more reliable than predicting the *exact return* (regression). Daily returns are extremely noisy; direction has more signal.

### Models We'll Compare
1. **Logistic Regression**: Linear baseline, interpretable coefficients
2. **Random Forest**: Non-linear, handles feature interactions, robust to outliers
3. **Gradient Boosting**: Sequential ensemble, typically best performance

### Important Caveat on Time-Series CV
Standard k-fold cross-validation would cause **data leakage** — using future data to predict the past. We use a simple train/test split with temporal ordering (no shuffling) to maintain time causality.

In [ ]:
# ── Define Feature Columns ────────────────────────────────────────────────────
SENTIMENT_FEATURES = [
    'mean_sentiment_score', 'max_sentiment_score', 'min_sentiment_score',
    'std_sentiment_score', 'mean_positive_prob', 'mean_negative_prob',
    'positive_ratio', 'negative_ratio', 'sentiment_balance',
    'rolling_sentiment_3d', 'rolling_sentiment_7d', 'rolling_sentiment_14d',
    'sentiment_lag_1d', 'sentiment_lag_2d', 'sentiment_lag_3d',
    'n_headlines',
]

TECHNICAL_FEATURES = [
    'return_1d', 'return_3d', 'return_7d', 'return_14d',
    'return_lag_1', 'return_lag_2', 'return_lag_3', 'return_lag_5',
    'price_vs_sma_3', 'price_vs_sma_7', 'price_vs_sma_14', 'price_vs_sma_30',
    'macd', 'macd_hist',
    'volatility_7d', 'volatility_14d', 'volatility_30d',
    'rsi_14', 'bb_position',
    'volume_ratio', 'volume_return',
    'hl_range', 'overnight_gap',
]

ALL_FEATURES = SENTIMENT_FEATURES + TECHNICAL_FEATURES

# ── Prepare Dataset ───────────────────────────────────────────────────────────
model_df = master_df.dropna(subset=['label'] + ['return_1d', 'rsi_14']).copy()

# Only keep rows with sentiment data
model_df = model_df.dropna(subset=['mean_sentiment_score'])

# Encode labels
le = LabelEncoder()
model_df['label_encoded'] = le.fit_transform(model_df['label'])
print(f'Label encoding: {dict(zip(le.classes_, le.transform(le.classes_)))}')

# Sort by date (CRITICAL for time-series — no future leakage)
model_df = model_df.sort_values('date').reset_index(drop=True)

# Feature matrix and target
available_features = [f for f in ALL_FEATURES if f in model_df.columns]
X = model_df[available_features]
y = model_df['label_encoded']

# Temporal train/test split (no shuffle!)
split_idx = int(len(model_df) * (1 - CONFIG['test_size']))
X_train, X_test = X.iloc[:split_idx], X.iloc[split_idx:]
y_train, y_test = y.iloc[:split_idx], y.iloc[split_idx:]

train_end = model_df['date'].iloc[split_idx - 1].date()
test_start = model_df['date'].iloc[split_idx].date()

print(f'\n📊 Dataset Summary:')
print(f'  Total samples: {len(model_df):,}')
print(f'  Features used: {len(available_features)}')
print(f'  Train: {len(X_train):,} samples ({CONFIG["start_date"]} → {train_end})')
print(f'  Test:  {len(X_test):,} samples ({test_start} → {model_df["date"].max().date()})')
print(f'\n  Label distribution in test set:')
for enc, label in zip(le.transform(le.classes_), le.classes_):
    count = (y_test == enc).sum()
    print(f'    {label}: {count} ({100*count/len(y_test):.1f}%)')

In [ ]:
# ── Train Multiple Models ──────────────────────────────────────────────────────

models = {
    'Logistic Regression': Pipeline([
        ('imputer', SimpleImputer(strategy='median')),
        ('scaler', StandardScaler()),
        ('clf', LogisticRegression(
            max_iter=1000, C=1.0, random_state=CONFIG['random_state']
        ))
    ]),
    'Random Forest': Pipeline([
        ('imputer', SimpleImputer(strategy='median')),
        ('clf', RandomForestClassifier(
            n_estimators=200, max_depth=8, min_samples_leaf=5,
            random_state=CONFIG['random_state'], n_jobs=-1
        ))
    ]),
    'Gradient Boosting': Pipeline([
        ('imputer', SimpleImputer(strategy='median')),
        ('clf', GradientBoostingClassifier(
            n_estimators=200, max_depth=4, learning_rate=0.05,
            subsample=0.8, random_state=CONFIG['random_state']
        ))
    ]),
}

results = {}
predictions = {}

print('🏋️ Training models...\n')

for name, pipeline in models.items():
    print(f'  Training {name}...')
    pipeline.fit(X_train, y_train)
    y_pred = pipeline.predict(X_test)
    y_prob = pipeline.predict_proba(X_test)
    
    acc = accuracy_score(y_test, y_pred)
    
    # One-vs-rest AUC for multi-class
    try:
        auc = roc_auc_score(y_test, y_prob, multi_class='ovr')
    except:
        auc = None
    
    results[name] = {
        'accuracy': acc,
        'auc': auc,
        'report': classification_report(y_test, y_pred, target_names=le.classes_, output_dict=True),
        'confusion_matrix': confusion_matrix(y_test, y_pred),
    }
    predictions[name] = y_pred
    
    auc_str = f'{auc:.4f}' if auc else 'N/A'
    print(f'    ✅ Accuracy: {acc:.4f} | AUC: {auc_str}')

# Best model
best_model_name = max(results, key=lambda k: results[k]['accuracy'])
best_model = models[best_model_name]
print(f'\n🏆 Best Model: {best_model_name} (Accuracy: {results[best_model_name]["accuracy"]:.4f})')

In [ ]:
# ── Model Comparison Chart ────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(18, 6))
fig.patch.set_facecolor(COLORS['bg'])
fig.suptitle('📊 Model Performance Comparison', fontsize=16, fontweight='bold', color='white')

model_names = list(results.keys())
bar_colors = ['#7b68ee', '#00d4aa', '#f5a623']

# Accuracy
accs = [results[n]['accuracy'] for n in model_names]
bars = axes[0].bar(model_names, accs, color=bar_colors, edgecolor='#0f1117', linewidth=1.5)
axes[0].axhline(y=1/3, color='white', linestyle='--', alpha=0.5, label='Random baseline')
axes[0].set_ylim(0, 1)
axes[0].set_title('Test Accuracy', color='white', fontweight='bold')
axes[0].set_ylabel('Accuracy')
axes[0].legend(fontsize=9)
for bar, acc in zip(bars, accs):
    axes[0].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01,
                 f'{acc:.3f}', ha='center', va='bottom', color='white', fontweight='bold')
axes[0].set_xticklabels(model_names, rotation=15, ha='right')

# AUC
aucs = [results[n]['auc'] or 0 for n in model_names]
bars = axes[1].bar(model_names, aucs, color=bar_colors, edgecolor='#0f1117', linewidth=1.5)
axes[1].set_ylim(0, 1)
axes[1].set_title('ROC-AUC (OvR)', color='white', fontweight='bold')
axes[1].set_ylabel('AUC')
for bar, auc in zip(bars, aucs):
    axes[1].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01,
                 f'{auc:.3f}', ha='center', va='bottom', color='white', fontweight='bold')
axes[1].set_xticklabels(model_names, rotation=15, ha='right')

# Per-class F1 (best model)
report = results[best_model_name]['report']
classes = le.classes_
f1_scores = [report[cls]['f1-score'] for cls in classes]
class_colors = [COLORS['down'], COLORS['neutral'], COLORS['up']]
bars = axes[2].bar(classes, f1_scores, color=class_colors, edgecolor='#0f1117', linewidth=1.5)
axes[2].set_ylim(0, 1)
axes[2].set_title(f'Per-Class F1 ({best_model_name})', color='white', fontweight='bold')
axes[2].set_ylabel('F1 Score')
for bar, f1 in zip(bars, f1_scores):
    axes[2].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01,
                 f'{f1:.3f}', ha='center', va='bottom', color='white', fontweight='bold')

for ax in axes:
    ax.grid(True, alpha=0.3, axis='y')

plt.tight_layout()
plt.savefig('model_comparison.png', dpi=120, bbox_inches='tight', facecolor=COLORS['bg'])
plt.show()

In [ ]:
# ── Confusion Matrix ──────────────────────────────────────────────────────────
fig, axes = plt.subplots(1, len(models), figsize=(18, 5))
fig.patch.set_facecolor(COLORS['bg'])
fig.suptitle('🎯 Confusion Matrices', fontsize=16, fontweight='bold', color='white')

for ax, (name, result) in zip(axes, results.items()):
    cm = result['confusion_matrix']
    cm_pct = cm.astype('float') / cm.sum(axis=1)[:, np.newaxis]
    
    im = ax.imshow(cm_pct, interpolation='nearest', cmap='Blues', vmin=0, vmax=1)
    
    tick_marks = np.arange(len(le.classes_))
    ax.set_xticks(tick_marks)
    ax.set_xticklabels(le.classes_, rotation=45, ha='right', color='white')
    ax.set_yticks(tick_marks)
    ax.set_yticklabels(le.classes_, color='white')
    
    # Annotate cells
    for i in range(cm.shape[0]):
        for j in range(cm.shape[1]):
            ax.text(j, i, f'{cm[i,j]}\n({cm_pct[i,j]:.0%})',
                    ha='center', va='center',
                    color='white' if cm_pct[i,j] < 0.5 else '#0f1117',
                    fontsize=10, fontweight='bold')
    
    star = ' ⭐' if name == best_model_name else ''
    ax.set_title(f'{name}{star}\nAcc: {result["accuracy"]:.3f}',
                 color='white', fontweight='bold')
    ax.set_ylabel('True Label' if ax == axes[0] else '', color='white')
    ax.set_xlabel('Predicted Label', color='white')

plt.tight_layout()
plt.savefig('confusion_matrices.png', dpi=120, bbox_inches='tight', facecolor=COLORS['bg'])
plt.show()

In [ ]:
# ── Feature Importance (Random Forest / Gradient Boosting) ────────────────────
fig, axes = plt.subplots(1, 2, figsize=(18, 8))
fig.patch.set_facecolor(COLORS['bg'])
fig.suptitle('🔍 Feature Importance Analysis', fontsize=16, fontweight='bold', color='white')

for ax, model_name in zip(axes, ['Random Forest', 'Gradient Boosting']):
    clf = models[model_name].named_steps['clf']
    importances = clf.feature_importances_
    
    feat_imp = pd.Series(importances, index=available_features)
    feat_imp = feat_imp.sort_values(ascending=True).tail(20)
    
    # Color by feature type
    colors = [
        COLORS['up'] if f in SENTIMENT_FEATURES else '#7b68ee'
        for f in feat_imp.index
    ]
    
    bars = ax.barh(feat_imp.index, feat_imp.values, color=colors)
    ax.set_title(f'{model_name}\nTop 20 Features', color='white', fontweight='bold')
    ax.set_xlabel('Importance')
    ax.grid(True, alpha=0.3, axis='x')
    
    # Legend
    sentiment_patch = mpatches.Patch(color=COLORS['up'], label='Sentiment Feature')
    technical_patch = mpatches.Patch(color='#7b68ee', label='Technical Feature')
    ax.legend(handles=[sentiment_patch, technical_patch], loc='lower right', fontsize=9)

plt.tight_layout()
plt.savefig('feature_importance.png', dpi=120, bbox_inches='tight', facecolor=COLORS['bg'])
plt.show()

# Print top features
clf = models['Gradient Boosting'].named_steps['clf']
feat_imp_gb = pd.Series(clf.feature_importances_, index=available_features).sort_values(ascending=False)
print('\n🏆 Top 15 Features (Gradient Boosting):')
for i, (feat, imp) in enumerate(feat_imp_gb.head(15).items()):
    type_tag = '📰 SENTIMENT' if feat in SENTIMENT_FEATURES else '📈 TECHNICAL'
    print(f'  {i+1:2d}. {type_tag} | {feat:35s}: {imp:.4f}')

## 💡 Section 10: Interpreting Results & Limitations

### What Good Performance Looks Like

Stock price prediction is famously difficult. Here's how to interpret your results:

| Accuracy | Interpretation |
|----------|----------------|
| ~33% | Random (baseline for 3 classes) |
| 40-45% | Weak but potentially useful signal |
| 45-55% | Moderate signal — consistent with academic literature |
| >60% | Suspiciously high — check for data leakage! |

### Key Limitations

1. **Synthetic news**: Real news has more complex patterns, breaking news events, and macro context
2. **Market efficiency**: Professional traders react to news in milliseconds; by close-of-day, most signal is priced in
3. **Survivorship bias**: We only analyze stocks that still exist (not failed companies)
4. **Transaction costs**: Even a model with edge may be unprofitable after spreads, slippage, and taxes
5. **Regime changes**: Models trained on 2023 data may fail in different market regimes
6. **Look-ahead bias risk**: Always verify your feature computation doesn't accidentally use future data

### Research Applications
This framework is valid for:
- Academic research on information efficiency
- Risk management (sentiment as a volatility predictor)
- Screening tool (flagging unusual sentiment shifts for human review)
- Building intuition about NLP × finance

In [ ]:
# ── Print Full Classification Report ──────────────────────────────────────────
print('=' * 70)
print(f'📊 FULL CLASSIFICATION REPORT — {best_model_name}')
print('=' * 70)

y_pred_best = predictions[best_model_name]
print(classification_report(
    y_test, y_pred_best,
    target_names=le.classes_
))

print('\n⚠️  RESEARCH DISCLAIMER')
print('-' * 70)
print('These results are for educational purposes only.')
print('Do NOT make investment decisions based on this model.')
print('Past pattern ≠ Future performance. Markets are non-stationary.')
print('=' * 70)

In [ ]:
# ── Save Model & Artifacts ────────────────────────────────────────────────────
import pickle

# Save the best model pipeline
model_artifacts = {
    'model': best_model,
    'label_encoder': le,
    'feature_names': available_features,
    'sentiment_features': SENTIMENT_FEATURES,
    'technical_features': TECHNICAL_FEATURES,
    'config': CONFIG,
    'best_model_name': best_model_name,
    'results_summary': {
        name: {'accuracy': r['accuracy'], 'auc': r['auc']}
        for name, r in results.items()
    }
}

with open('sentiment_model.pkl', 'wb') as f:
    pickle.dump(model_artifacts, f)

print('✅ Model artifacts saved to sentiment_model.pkl')
print(f'   Best model: {best_model_name}')
print(f'   Test accuracy: {results[best_model_name]["accuracy"]:.4f}')
print(f'   Features: {len(available_features)}')

# Final summary
print('\n' + '=' * 60)
print('🎓 NOTEBOOK COMPLETE — Key Takeaways:')
print('=' * 60)
print('1. FinBERT provides domain-adapted sentiment scores for financial text')
print('2. Combining NLP features with technical indicators improves signal')
print('3. Gradient Boosting typically outperforms linear models on this task')
print('4. Feature importance reveals which signals matter most')
print('5. Time-series CV is CRITICAL to avoid look-ahead bias')
print('\n→ Launch the Streamlit app for interactive exploration!')
print('  streamlit run app.py')